In [1]:
import requests
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
url = "http://ws.audioscrobbler.com/2.0/"
API_KEY = "bd5cff0c8431c3b8888f8db83ed41ace"

df = pd.read_csv("data/songs.csv")
df.head()
first_10000 = df.head(10000)


In [22]:
def musicbrainz_genres(artist_name):
    url = f"https://musicbrainz.org/ws/2/artist/?query=artist:{artist_name}&fmt=json"
    data = requests.get(url).json()
    artists = data.get("artists", [])
    if artists:
        return [tag["name"] for tag in artists[0].get("tags", [])]
    return []

In [18]:
def getLabel(artist_name, track_name, index):
    params = {
        "method": "track.getInfo",
        "artist": artist_name,
        "track": track_name,
        "api_key": API_KEY,
        "method": "getTopTags",
        "format": "json"
    }
    response = requests.get(url, params=params)
    track_info = response.json()
    tags =[]
    if "track" in track_info and "toptags" in track_info["track"]:
        tags = [tag["name"] for tag in track_info["track"]["toptags"].get("tag", [])]
    if(len(tags) != 0):
        first_10000.loc[index, 'tags'] = ", ".join(tags)
    # else:
    #     tags = musicbrainz_genres(artist_name)
    #     time.sleep(1)
    print(response.text)
    print(f"{index}, {artist_name}, {track_name}, {tags}")
    time.sleep(.1)
        

In [ ]:
for r in first_10000.itertuples():
    getLabel(r.artist, r.track, r.Index)

In [ ]:
(lambda r: getLabel(r.artist, r.track, r.Index), rows)

In [ ]:
rows = list(df.itertuples())

with ThreadPoolExecutor(max_workers=5) as executor:
    executor.map(lambda r: getLabel(r.artist, r.track, r.Index), rows)

In [9]:
count = 0
for item in first_10000['tags']:
    if(pd.isna(item)):
        count = count+1
print(count)

5851


In [24]:
for r in first_10000.itertuples():
    if(r.Index < 1500 and pd.isna(r.tags)): 
        first_10000.loc[r.Index, 'tags'] =", ".join(musicbrainz_genres(r.artist))
        print(first_10000.loc[r.Index, 'tags'])
        time.sleep(1)

pop, rap, jazz, dance, house, hip hop, funk, disco, gangsta rap, pop rap, neo soul, alternative hip hop, experimental hip hop, west coast hip hop, boom bap, r&b, 2010s, jazz rap, conscious hip hop, political hip hop, hardcore hip hop, trap, future bass, relic inn, 2020s, progressive rap, contemporary rap, conscious rapper
pop, drum and bass, jungle, dance-pop, alternative pop, uk garage, future garage, bubblegum pop, 2020s, gen z
icon, barbadian, pop, contemporary r&b, dance-pop, dancehall, electropop, dance, electro house, pop soul, reggae, pop star, female vocal, pop culture, r&b, influential pop star, barbadien, barbade, contemporary r b, hip hop, pop and chart, rnb, rihanna, rehab, barbados
electronic, synthpop, pop, jazz, dance, american, usa, synth-pop, electropop, disco, americana, américain, female vocals, 2000s, dance-pop, adult contemporary, pop rock, europop, electro house, film soundtrack, united states, britney spears, the fame, gata rahe mera dil, 2010s, art pop, edge of 

In [ ]:
for r in df.itertuples():
    df.loc[r.Index, 'ratio'] = r.listeners/r.playcount

In [25]:
first_10000.dropna()

,artist,track,listeners,playcount,tags
0,The Weeknd,Blinding Lights,2314795,37860617,"synthwave, synthpop, pop, 2019, 2010s"
1,The Weeknd,Starboy,2297163,32833915,"rnb, pop, electronic, Daft Punk, 2016"
2,The Weeknd,The Hills,2084094,27009303,"rnb, electronic, trap, r&b, love at first listen"
3,The Weeknd,Die for You,1954869,33126883,"rnb, pop, Ballad, synthpop, Canadian"
4,The Weeknd,Can't Feel My Face,1826293,18326029,"rnb, pop, love at first listen, r&b, Disco"
...,...,...,...,...,...
9981,Cris MJ,SI NO ES CONTIGO,238369,3640860,latin
9990,Audrey Hobert,Sue me,195408,3116865,"female vocalist, pop, electronic, electropop, hot"
9991,Audrey Hobert,Bowling alley,114063,1513795,"pop, indie, indie pop, dance-pop, dariacore"
9994,Audrey Hobert,Shooting star,75712,978702,"indie, synthpop, United States, indie pop, mil..."


In [26]:
first_10000.to_csv("data/songs_with_tags.csv", index=False)